# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (run once per environment)
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display basic metadata and description
meta = dataset.metadata
print(f"Title: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields for exploration.

In [ ]:
# List all record sets in the dataset (by @id and name)
print("Available record sets (by @id):\n")
all_record_sets = list(dataset.metadata.record_sets)

for rset in all_record_sets:
    print(f"- @id: {rset['@id']}")
    print(f"  Name: {rset.get('name', 'N/A')}")
    print(f"  Description: {rset.get('description', 'N/A')}")
    
    # List fields within each record set (by @id and name)
    if 'fields' in rset:
        print("  Fields:")
        for field in rset['fields']:
            print(f"    - @id: {field['@id']}, name: {field.get('name','N/A')} (type: {field.get('dataType','?')})")
    print('')

# Save list of record set @id's
record_set_ids = [rset['@id'] for rset in all_record_sets]

## 3. Data Extraction
Load records from a selected record set into a pandas DataFrame for further analysis. Choose the record set `@id` of interest (for mass spec, look for main protein/analysis records).

In [ ]:
# For this dataset, we'll inspect all record sets
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records in this record set
    print(f"Loading records from record set: {record_set_id}")
    try:
        recs = list(dataset.records(record_set=record_set_id))
        if recs:
            df = pd.DataFrame(recs)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  [No records found]")
    except Exception as e:
        print(f"  [Failed to load: {e}]")
    print("")

# We select the main protein-level record set for demo. Manually, we look for the right @id, or take first if only one.
main_record_set_id = None
for rs in dataframes.keys():
    # As an example, look for 'protein' in the id
    if 'protein' in rs.lower():
        main_record_set_id = rs
        break
if not main_record_set_id:
    main_record_set_id = record_set_ids[0]  # fall back to first
print(f"Main record set used for EDA: {main_record_set_id}")
display_cols = dataframes[main_record_set_id].columns.tolist()
print('Columns:', display_cols)
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization and grouping.

Let's:
- Select a numeric field (e.g., `coverage_percent`, `molecular_weight`, etc. by @id/column name)
- Demonstrate filtering (e.g., MW > 50000), normalization, and grouping (e.g., by sample/condition if present)
- Reference fields by `@id` if possible (otherwise column name in the DataFrame)

In [ ]:
# Inspect columns to pick a numeric field
df = dataframes[main_record_set_id]
numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
print("Numeric fields in this record set:", numeric_fields)

# Pick a numeric field: try 'molecular_weight' (or similar), else use the first numeric field
field_choices = [c for c in numeric_fields if 'weight' in c.lower() or 'mw' in c.lower() or 'score' in c.lower() or 'abundance' in c.lower() or 'coverage' in c.lower()]
if field_choices:
    numeric_field = field_choices[0]
else:
    numeric_field = numeric_fields[0] if numeric_fields else None
print(f"Selected numeric field: {numeric_field}")

# Filtering: e.g., MW > 60000 or abundance > threshold. We'll use mean as a threshold for demo
if numeric_field:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} rows.")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Group by a key (try to use 'sample', 'condition', or any string field with <10 cardinality)
    group_field = None
    for c in df.columns:
        if c != numeric_field and df[c].dtype == object:
            nvals = df[c].nunique()
            if 1 < nvals < 10:
                group_field = c
                break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df)
    else:
        print("No suitable group field found.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions of numeric fields, for example histograms or boxplots.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # Boxplot if there's a group_field
    if group_field:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to discover and analyze a mass spectrometry dataset defined by the FAIR^2 Croissant schema.

**Key findings:**
- The primary record set contains protein-level data with fields such as molecular weight, abundance, and possibly sample condition/group.
- Basic numeric EDA, normalization, and visualization were demonstrated using field `@id` as a reference.
- The Croissant schema enables easy referencing of datasets and fields using `@id`, supporting reproducible FAIR workflows.

_Tip: Explore the `fields` info in the Croissant metadata for more complex queries or advanced feature engineering!_